# Data Cleaning & Feature Engineering — Code Episodes
### 14 real, runnable programs from the 24-episode course

This course turns the raw **Telco Customer Churn** dataset (7,043 rows × 21 columns, 26.5% churn) into a clean, feature-engineered, model-ready dataset — with strict ML hygiene woven through every step: **split before you clean, fit only on train, transform test with what train learned.**

Every episode in the course runs real pandas/sklearn code. **14 of the 24 per-episode scripts survived on disk** after the build — the other 10 were generated, run once for their terminal output and figure, and not persisted. This notebook ships exactly the 14 verified real scripts, unmodified except for the CSV path:

| # | Episode | What it shows |
|---|---|---|
| DC04 | Data Types | coerce TotalCharges, right-type categoricals |
| DC06 | Missing Values | SimpleImputer fit-on-train |
| DC07 | Duplicates & Text | standardize inconsistent category strings |
| DC09 | Leaky Features | the leakage test on customerID |
| DC11 | Numerical FE | ratios & interactions |
| DC13 | Skewed Features | log1p vs Yeo-Johnson |
| DC15 | One-Hot Encoding | fit-on-train, transform-test |
| DC16 | Ordinal & Frequency | order-aware vs count-based encoding |
| DC17 | Target Encoding | naive (leaky) vs smoothed vs out-of-fold |
| DC18 | Feature Scaling | MinMax vs Standard vs Robust |
| DC19 | Correlation | multicollinearity & VIF |
| DC20 | Feature Selection | variance / F-score / RF importance |
| DC21 | Pipelines | chained impute→scale, no leakage |
| DC22 | ColumnTransformer | numeric + categorical branches glued back |

Run the setup cell once (installs the libraries + fetches the dataset), then each episode cell runs top-to-bottom.

*— @kader_mohideen*

### Setup
Install the libraries the episodes use. `telco_churn.csv` (the canonical Telco Customer Churn dataset, 7,043×21) is committed alongside this notebook — no download needed.

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib

## DC04 — Data Types — Identify & Fix

TotalCharges is stored as text with 11 blank cells (all tenure==0). Coerce to numeric, right-type the categoricals, and cut memory 78% (1909 KB → 421 KB).

In [ ]:
# dc04 -- fix dtypes: TotalCharges text->numeric (11 NaN), categoricals, memory win.
import pandas as pd
import matplotlib.pyplot as plt
import os

raw = pd.read_csv("telco_churn.csv")
print("TotalCharges dtype (before):", raw["TotalCharges"].dtype)
mem_before = raw.memory_usage(deep=True).sum() / 1024

df = raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
n_nan = df["TotalCharges"].isna().sum()
print("TotalCharges dtype (after):", df["TotalCharges"].dtype, "| NaN created:", n_nan)

print("all NaN rows have tenure==0:",
      bool((df.loc[df["TotalCharges"].isna(), "tenure"] == 0).all()))

df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"}).astype("category")
for c in df.select_dtypes(include=["object"]).columns:
    df[c] = df[c].astype("category")
mem_after = df.memory_usage(deep=True).sum() / 1024
print(f"memory: {mem_before:.0f} KB -> {mem_after:.0f} KB "
      f"({(1 - mem_after/mem_before)*100:.0f}% smaller)")

os.makedirs("remotion/public/cleanfig", exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["before", "after"], [mem_before, mem_after], color=["#7d8693", "#7cc1ff"])
ax.set_ylabel("memory (KB)")
ax.set_title(f"Right-typed: TotalCharges numeric, {n_nan} NaN", color="#7cc1ff")
fig.savefig("remotion/public/cleanfig/dc04.png", dpi=120, bbox_inches="tight")
print("saved remotion/public/cleanfig/dc04.png")


## DC06 — Handling Missing Values

SimpleImputer fit on train only (median 1398.12), then applied to both train and test — the fit/transform split that keeps test data honest.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

df = pd.read_csv("telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("missing TotalCharges:", int(df["TotalCharges"].isna().sum()))

X = df.drop(columns=["Churn"]); y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

imp = SimpleImputer(strategy="median")
imp.fit(X_train[["TotalCharges"]])

X_train["TotalCharges"] = imp.transform(X_train[["TotalCharges"]])
X_test["TotalCharges"]  = imp.transform(X_test[["TotalCharges"]])
print("train median used:", round(float(imp.statistics_[0]), 2))
print("missing after fill (train):", int(X_train["TotalCharges"].isna().sum()))
print("missing after fill (test):", int(X_test["TotalCharges"].isna().sum()))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(X_train["TotalCharges"], bins=40, color="#ffd166", alpha=0.85)
ax.axvline(imp.statistics_[0], color="#ff6b6b", lw=2, label="train median fill")
ax.set_title("TotalCharges after median imputation"); ax.legend()
fig.savefig("remotion/public/cleanfig/dc06.png", dpi=130, bbox_inches="tight")
print("saved dc06.png")


## DC07 — Duplicates & Text Cleaning

Zero duplicate rows, but 'No' vs 'No internet service' vs 'No phone service' are the same answer in three costumes — standardize them before they poison an encoder.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("telco_churn.csv")
# coerce TotalCharges to numeric (11 blanks -> NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
X = df.drop(columns=["Churn"]); y = df["Churn"]
X_train, X_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)[:2]

# b1 duplicate check
print("duplicate rows (train):", int(X_train.duplicated().sum()))

def clean_text(frame):
    f = frame.copy()
    obj = f.select_dtypes("object").columns
    # b2 strip whitespace + unify case
    for c in obj:
        f[c] = f[c].str.strip()
    # b3 collapse "no service" phrasings to "No"
    f[obj] = f[obj].replace(
        {"No internet service": "No", "No phone service": "No"})
    return f

# b4 before/after for OnlineSecurity
before = X_train["OnlineSecurity"].value_counts().to_dict()
X_train = clean_text(X_train)
print("OnlineSecurity before:", before)
print("OnlineSecurity after :", X_train["OnlineSecurity"].value_counts().to_dict())

# b5 same mapping on test
X_test = clean_text(X_test)
print("test OnlineSecurity  :", X_test["OnlineSecurity"].value_counts().to_dict())

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
vc = X_train["OnlineSecurity"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(vc.index, vc.values, color="#ffd166")
ax.set_title("OnlineSecurity after standardizing to Yes / No")
ax.set_ylabel("count")
fig.savefig("remotion/public/cleanfig/dc07.png", dpi=130, bbox_inches="tight")
print("saved dc07.png")


## DC09 — Dropping Irrelevant & Leaky Features

customerID is 100% unique (5634/5634) — a model can memorize it instead of learning. The leakage test: would I have this value BEFORE the outcome?

In [ ]:
import pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

df = pd.read_csv("telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
from sklearn.model_selection import train_test_split
X = df.drop(columns=["Churn"]); y = df["Churn"]
X_train, X_test = train_test_split(X, test_size=0.20, random_state=42, stratify=y)[:2]

# b1 customerID is unique per row
print("customerID unique values:", X_train["customerID"].nunique(), "of", len(X_train))

# b2 drop from train
drop_cols = ["customerID"]
X_train = X_train.drop(columns=drop_cols)
# b3 drop from test
X_test = X_test.drop(columns=drop_cols)

# b4 confirm shapes
print("train cols:", X_train.shape[1], "test cols:", X_test.shape[1])
print("columns match:", X_train.columns.equals(X_test.columns))

# b5 figure: keep/drop ledger
keep = list(X_train.columns)
drop = drop_cols
fig, ax = plt.subplots(figsize=(6, 5))
fig.patch.set_facecolor("white")
ax.axis("off")
ax.text(0.02, 0.95, f"KEEP ({len(keep)})", color="#9fd97a", weight="bold", fontsize=14, transform=ax.transAxes)
for i, c in enumerate(keep):
    ax.text(0.02, 0.88 - i*0.045, c, color="#333", fontsize=9, transform=ax.transAxes)
ax.text(0.65, 0.95, f"DROP ({len(drop)})", color="#ff6b6b", weight="bold", fontsize=14, transform=ax.transAxes)
ax.text(0.65, 0.88, "customerID\n(unique id — no signal)", color="#333", fontsize=9, transform=ax.transAxes)
ax.set_title("Feature keep / drop ledger", fontsize=13, color="#bd7dff", pad=10)
plt.tight_layout()
plt.savefig("remotion/public/cleanfig/dc09.png", dpi=130, bbox_inches="tight")
print("figure saved: dc09.png")


## DC11 — Numerical Feature Engineering — Ratios & Interactions

expected_total = tenure × MonthlyCharges (an interaction); charge_consistency = TotalCharges / expected_total (a ratio) — both row-wise safe, no leakage.

In [ ]:
# dc11 - numerical FE: interaction & ratio; check TotalCharges vs tenure*MonthlyCharges.
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("telco_churn.csv").drop(columns=["customerID"])
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
y = (df["Churn"] == "Yes").astype(int)
tr, te = train_test_split(df, test_size=0.2, stratify=y, random_state=42)

def feats(d):
    d = d.copy()
    d["expected_total"] = d["tenure"] * d["MonthlyCharges"]
    d["charge_consistency"] = d["TotalCharges"] / d["expected_total"].replace(0, np.nan)
    return d

tr, te = feats(tr), feats(te)
med = tr["charge_consistency"].median()
corr = tr[["TotalCharges","expected_total"]].corr().iloc[0,1]
print("expected_total = tenure * MonthlyCharges (interaction)")
print("charge_consistency = TotalCharges / expected_total (ratio)")
print("train median consistency: %.3f" % med)
print("corr(TotalCharges, expected_total) train: %.3f" % corr)
print("rows with tenure==0 (consistency NaN) train:", int((tr["expected_total"]==0).sum()))

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,4)); fig.patch.set_facecolor("white")
ax.scatter(tr["expected_total"], tr["TotalCharges"], s=6, alpha=0.3, color="#bd7dff")
lim = [0, float(tr["expected_total"].max())]
ax.plot(lim, lim, color="#ffd166", lw=2, label="y = x (perfect)")
ax.set_xlabel("expected_total = tenure x MonthlyCharges"); ax.set_ylabel("actual TotalCharges")
ax.set_title("Actual vs expected TotalCharges (train)", color="#bd7dff"); ax.legend()
plt.tight_layout(); plt.savefig("remotion/public/cleanfig/dc11.png", dpi=120)
print("figure saved: dc11.png")


## DC13 — Handling Skewed Features

TotalCharges has skew +0.95. log1p pulls it to -0.82; PowerTransformer (Yeo-Johnson), fit on train only, gets it to -0.15 on train and -0.14 on test.

In [ ]:
# dc13 - skewed features: log1p & PowerTransformer (Yeo-Johnson), FIT ON TRAIN ONLY.
import numpy as np, pandas as pd
from scipy.stats import skew
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer

df = pd.read_csv("telco_churn.csv").drop(columns=["customerID"])
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
y = (df["Churn"] == "Yes").astype(int)
tr, te = train_test_split(df, test_size=0.2, stratify=y, random_state=42)

raw_skew = skew(tr["TotalCharges"])
log_skew = skew(np.log1p(tr["TotalCharges"]))
pt = PowerTransformer(method="yeo-johnson")
pt.fit(tr[["TotalCharges"]])
yj_tr = pt.transform(tr[["TotalCharges"]]).ravel()
yj_te = pt.transform(te[["TotalCharges"]]).ravel()

print("TotalCharges skew (train, raw):        %.3f" % raw_skew)
print("TotalCharges skew (train, log1p):      %.3f" % log_skew)
print("TotalCharges skew (train, Yeo-Johnson):%.3f" % skew(yj_tr))
print("learned lambda (train):", round(pt.lambdas_[0], 3))
print("test skew after train-fitted YJ:      %.3f" % skew(yj_te))

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(7,3.4)); fig.patch.set_facecolor("white")
axes[0].hist(tr["TotalCharges"], bins=40, color="#ff6b6b", alpha=0.7)
axes[0].set_title("raw  skew=%.2f" % raw_skew, color="#ff6b6b")
axes[1].hist(yj_tr, bins=40, color="#9fd97a", alpha=0.8)
axes[1].set_title("Yeo-Johnson  skew=%.2f" % skew(yj_tr), color="#9fd97a")
for a in axes: a.set_xlabel("TotalCharges"); a.set_ylabel("count")
fig.suptitle("Taming skew, fitted on train", color="#bd7dff")
plt.tight_layout(); plt.savefig("remotion/public/cleanfig/dc13.png", dpi=120)
print("saved: remotion/public/cleanfig/dc13.png")


## DC15 — Categorical Encoding I — One-Hot

15 nominal columns become 26 one-hot columns (drop='first', handle_unknown='ignore') — fit on train, the same 26-wide transform applied to test.

In [ ]:
# DC15 · One-Hot Encoding — fit on TRAIN only, drop='first', handle_unknown='ignore'
import pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("telco_churn.csv").drop(columns=["customerID"])
y = df["Churn"]; X = df.drop(columns=["Churn"])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

nominal = ["gender","Partner","Dependents","PhoneService","MultipleLines",
           "InternetService","OnlineSecurity","OnlineBackup","DeviceProtection",
           "TechSupport","StreamingTV","StreamingMovies","Contract",
           "PaperlessBilling","PaymentMethod"]
ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
Otr = ohe.fit_transform(Xtr[nominal])
Ote = ohe.transform(Xte[nominal])

before = len(nominal)
after  = Otr.shape[1]
print("nominal cols in :", before)
print("one-hot cols out:", after)
print(f"train one-hot shape: {Otr.shape}  test one-hot shape: {Ote.shape}")
assert Otr.shape[1] == Ote.shape[1], "train/test width mismatch"
print("widths match (stackable):", Otr.shape[1] == Ote.shape[1])

# hero figure
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.bar(["before\n(words)","after\n(one-hot)"], [before, after],
       color=["#7cc1ff","#bd7dff"], edgecolor="#1b2330")
for i,v in enumerate([before, after]): ax.text(i, v+0.6, str(v), ha="center", fontweight="bold", fontsize=14)
ax.set_ylabel("column count")
ax.set_title(f"one-hot: {before} word columns -> {after} numeric columns")
fig.tight_layout(); fig.savefig("remotion/public/cleanfig/dc15.png", dpi=130)
print("saved: remotion/public/cleanfig/dc15.png")


## DC16 — Categorical Encoding II — Ordinal & Frequency

Contract has a real order (Month-to-month < One year < Two year) → OrdinalEncoder. PaymentMethod doesn't → frequency-encode it from train counts, reused on test.

In [ ]:
# DC16 · Ordinal (Contract) + Frequency (PaymentMethod) — fit/learn on TRAIN only
import pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

df = pd.read_csv("telco_churn.csv").drop(columns=["customerID"])
y = df["Churn"]; X = df.drop(columns=["Churn"])
Xtr, Xte = train_test_split(X, test_size=0.2, stratify=y, random_state=42)

order = ["Month-to-month", "One year", "Two year"]
oe = OrdinalEncoder(categories=[order])
Xtr = Xtr.copy(); Xte = Xte.copy()
Xtr["Contract_ord"] = oe.fit_transform(Xtr[["Contract"]]).astype(int)
Xte["Contract_ord"] = oe.transform(Xte[["Contract"]]).astype(int)

freq = Xtr["PaymentMethod"].value_counts(normalize=True)
Xtr["Pay_freq"] = Xtr["PaymentMethod"].map(freq)
Xte["Pay_freq"] = Xte["PaymentMethod"].map(freq).fillna(0.0)

print("Contract order map:", dict(zip(order, range(len(order)))))
print("PaymentMethod train freq:")
print(freq.round(3).to_string())
assert Xte["Pay_freq"].notna().all(), "test has unseen PaymentMethod"
print("test Pay_freq has no NaN:", Xte["Pay_freq"].notna().all())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4))
a1.bar(order, [0,1,2], color="#bd7dff", edgecolor="#1b2330")
a1.set_title("Contract -> ordinal (0,1,2)"); a1.tick_params(axis="x", rotation=20)
a2.bar(freq.index, freq.values, color="#7cc1ff", edgecolor="#1b2330")
a2.set_title("PaymentMethod -> frequency (train)"); a2.tick_params(axis="x", rotation=20)
fig.suptitle("ordinal keeps order  ·  frequency keeps width small")
fig.tight_layout(); fig.savefig("remotion/public/cleanfig/dc16.png", dpi=130)
print("saved: remotion/public/cleanfig/dc16.png")


## DC17 — Target Encoding — Naive vs Smoothed vs Out-of-Fold

Naive target-mean encoding leaks (the column can perfectly predict itself); smoothing toward the global rate and 5-fold out-of-fold encoding both fix it — compared side by side.

In [ ]:
# DC17 · Target Encoding — naive (LEAKY) vs smoothed vs out-of-fold; learn on TRAIN only
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold

df = pd.read_csv("telco_churn.csv").drop(columns=["customerID"])
df["y"] = (df["Churn"] == "Yes").astype(int)
Xtr, Xte = train_test_split(df, test_size=0.2, stratify=df["y"], random_state=42)
g = Xtr["y"].mean(); col = "PaymentMethod"
print("global churn (train):", round(float(g), 3))

naive_map = Xtr.groupby(col)["y"].mean()
Xtr["te_naive"] = Xtr[col].map(naive_map)

cnt = Xtr.groupby(col)["y"].count(); m = 50
smooth_map = (cnt * naive_map + m * g) / (cnt + m)
Xtr["te_smooth"] = Xtr[col].map(smooth_map)

Xtr["te_oof"] = np.nan
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for tr_i, va_i in kf.split(Xtr):
    fold = Xtr.iloc[tr_i].groupby(col)["y"].mean()
    Xtr.iloc[va_i, Xtr.columns.get_loc("te_oof")] = Xtr.iloc[va_i][col].map(fold).fillna(g).values
print("corr w/ y  naive:", round(float(Xtr["te_naive"].corr(Xtr["y"])), 3),
      " oof:", round(float(Xtr["te_oof"].corr(Xtr["y"])), 3))

Xte = Xte.copy(); Xte["te_smooth"] = Xte[col].map(smooth_map).fillna(g)
print("test te_smooth has no NaN:", bool(Xte["te_smooth"].notna().all()))

# Print per-category values
for cat in sorted(naive_map.index):
    oof_cat = Xtr[Xtr[col]==cat]["te_oof"].mean()
    print(f"  {cat:25s}  naive={naive_map[cat]:.3f}  smooth={smooth_map[cat]:.3f}  oof={oof_cat:.3f}")

cats = list(naive_map.index)
fig, ax = plt.subplots(figsize=(8.5, 3.6))
xpos = np.arange(len(cats))
ax.bar(xpos-0.25, naive_map.values,  width=0.25, color="#ff6b6b", label="naive (leaky)")
ax.bar(xpos,      smooth_map.values, width=0.25, color="#bd7dff", label="smoothed")
ax.bar(xpos+0.25, Xtr.groupby(col)["te_oof"].mean().reindex(cats).values, width=0.25, color="#7cc1ff", label="out-of-fold")
ax.axhline(float(g), color="#7d8693", ls="--", label="global rate")
ax.set_xticks(xpos); ax.set_xticklabels(cats, rotation=20); ax.legend()
ax.set_title("target encoding: naive leaks · smoothing + OOF stay honest")
fig.tight_layout(); fig.savefig("remotion/public/cleanfig/dc17.png", dpi=130)
print("saved: remotion/public/cleanfig/dc17.png")


## DC18 — Feature Scaling — Normalization vs Standardization

TotalCharges ranges 19–8685 while SeniorCitizen is 0/1 — an unscaled model lets the big number dominate. MinMax, StandardScaler, and RobustScaler compared on the real ranges.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df = pd.read_csv("telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
num = ["tenure", "MonthlyCharges", "TotalCharges"]
X_tr, X_te = train_test_split(df, test_size=0.2, stratify=df["Churn"], random_state=42)
print(f"train: {len(X_tr)}  test: {len(X_te)}")

scalers = {"MinMax": MinMaxScaler(), "Standard": StandardScaler(), "Robust": RobustScaler()}
out = {}
for name, sc in scalers.items():
    sc.fit(X_tr[num])
    out[name] = sc.transform(X_te[num])
for name, arr in out.items():
    print(f"{name:9s} TotalCharges  min={arr[:,2].min():7.3f}  max={arr[:,2].max():7.3f}  mean={arr[:,2].mean():7.3f}")

# Raw ranges for Act-1
for c in num:
    print(f"  {c}: min={X_tr[c].min():.1f}  max={X_tr[c].max():.1f}")

fig, ax = plt.subplots(1, 4, figsize=(13, 3))
ax[0].hist(X_te["TotalCharges"], bins=30, color="#34D6FF", edgecolor="#0a0e16")
ax[0].set_title("Raw TotalCharges", fontsize=10)
for i, name in enumerate(scalers):
    ax[i+1].hist(out[name][:, 2], bins=30, color="#34D6FF", edgecolor="#0a0e16")
    ax[i+1].set_title(name, fontsize=10)
for a in ax:
    a.set_facecolor("#f8f8f8")
fig.patch.set_facecolor("white")
fig.tight_layout()
fig.savefig("remotion/public/cleanfig/dc18.png", dpi=130)
print("Figure saved: dc18.png")


## DC19 — Correlation & Multicollinearity

tenure and TotalCharges correlate at r=0.83. Variance Inflation Factor (VIF) flags TotalCharges at 8.22 — the redundant signal a linear model would double-count.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df = pd.read_csv("telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
X_tr, X_te = train_test_split(df, test_size=0.2, stratify=df["Churn"], random_state=42)

# b2 correlation on TRAIN only
corr = X_tr[num].corr()
print("=== CORRELATION MATRIX (TRAIN) ===")
print(corr.to_string())

# b3 VIF on TRAIN
vif_vals = [variance_inflation_factor(X_tr[num].values, i) for i in range(len(num))]

# b4 strongest pair + VIF
pair = corr.where(~np.eye(len(num), dtype=bool)).abs().stack().idxmax()
print(f"\nstrongest pair: {pair[0]} ~ {pair[1]}  r={corr.loc[pair]:.2f}")
for c, v in zip(num, vif_vals):
    print(f"VIF  {c:15s} {v:6.2f}")

# b5 heatmap figure
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap="cividis", vmin=-1, vmax=1)
ax.set_xticks(range(len(num))); ax.set_xticklabels(num, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(num))); ax.set_yticklabels(num, fontsize=9)
for (i, j), z in np.ndenumerate(corr.values):
    ax.text(j, i, f"{z:.2f}", ha="center", va="center", color="w", fontsize=9)
fig.colorbar(im)
ax.set_title("Correlation (TRAIN)", fontsize=11)
fig.tight_layout()
fig.savefig("remotion/public/cleanfig/dc19.png", dpi=130)
print("\nFigure saved: dc19.png")
print(f"\ntrain shape: {X_tr.shape}")


## DC20 — Feature Selection Basics

Three lenses on the same features — variance threshold, SelectKBest (F-score), and Random Forest importance — and where they agree (and don't).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df = pd.read_csv("telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df["y"] = (df["Churn"] == "Yes").astype(int)
num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
X_tr, X_te, y_tr, y_te = train_test_split(df[num], df["y"], test_size=0.2,
                                          stratify=df["y"], random_state=42)

# variance threshold on train
vt = VarianceThreshold(0.0).fit(X_tr)
kept = list(X_tr.columns[vt.get_support()])
print(f"kept by variance: {kept}")

# selectkbest ANOVA F on train
skb = SelectKBest(f_classif, k="all").fit(X_tr, y_tr)

# random forest on train
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)

for c, f_score, imp in zip(num, skb.scores_, rf.feature_importances_):
    print(f"{c:15s} F={f_score:8.1f}   rf_importance={imp:.3f}")

# figure
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].barh(num, skb.scores_, color="#34D6FF")
ax[0].set_title("SelectKBest F-score")
ax[1].barh(num, rf.feature_importances_, color="#34D6FF")
ax[1].set_title("RandomForest importance")
fig.tight_layout()
fig.savefig("remotion/public/cleanfig/dc20.png", dpi=130)
print("figure saved")


## DC21 — scikit-learn Pipelines

Chain impute→scale into one Pipeline object. Fit once on train; the same fitted steps transform test with zero chance of accidentally refitting on it.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
num = ["tenure", "MonthlyCharges", "TotalCharges"]
X, y = df[num], df["Churn"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())])
pipe.fit(X_tr)
Z_te = pipe.transform(X_te)
learned_median = pipe.named_steps["impute"].statistics_[2]
print("train median TotalCharges:", round(learned_median, 2))
print("test transformed shape:", Z_te.shape, "| test mean ~0?:", round(Z_te.mean(), 3))

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(Z_te[:, 2], bins=30, color="#5CE08A", edgecolor="#07090f")
ax.set_title("Test TotalCharges after fitted pipeline (scaled)")
ax.set_xlabel("standardized value"); ax.set_ylabel("count")
fig.savefig("remotion/public/cleanfig/dc21.png", dpi=130, bbox_inches="tight")
print("figure saved")


## DC22 — ColumnTransformer

Route numeric columns to one pipeline branch and categorical columns to another, then glue the outputs back into one 13-wide model-ready matrix.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt

df = pd.read_csv("telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.drop(columns=["customerID"])

num = ["tenure", "MonthlyCharges", "TotalCharges"]
cat = ["Contract", "PaymentMethod", "InternetService"]
X, y = df[num + cat], df["Churn"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
    stratify=y, random_state=42)

num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                     ("ohe", OneHotEncoder(handle_unknown="ignore"))])
ct = ColumnTransformer([("num", num_pipe, num), ("cat", cat_pipe, cat)])
ct.fit(X_tr)
Ztr, Zte = ct.transform(X_tr), ct.transform(X_te)
print(f"train matrix: {Ztr.shape} | test matrix: {Zte.shape}")
print(f"raw input cols: {len(num + cat)} | after CT: {Ztr.shape[1]}")
cats = ct.named_transformers_["cat"].named_steps["ohe"].categories_
for name, cats_i in zip(cat, cats):
    print(f"  {name}: {list(cats_i)} ({len(cats_i)} levels)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["raw cols", "after CT"], [len(num + cat), Ztr.shape[1]],
       color=["#7d8693", "#5CE08A"], edgecolor="#07090f")
ax.set_title("Columns before vs after ColumnTransformer")
ax.set_ylabel("column count")
for i, v in enumerate([len(num + cat), Ztr.shape[1]]):
    ax.text(i, v + 0.3, str(v), ha="center", fontweight="bold", fontsize=14)
fig.savefig("remotion/public/cleanfig/dc22.png", dpi=130, bbox_inches="tight")
print("figure saved: dc22.png")


---
*Built from the real episode scripts. The full video course walks all 24 episodes in order — orient → clean → engineer → transform → pipeline — ending in a capstone that assembles one reusable `ColumnTransformer` pipeline. — @kader_mohideen*